# 🏗️ Notebook 3: OOP para Data Science — Clases, Métodos y Patrones
### Curso: Machine Learning Avanzado — Módulo 1

**Objetivo:** Dominar la Programación Orientada a Objetos aplicada a la construcción de pipelines, transformadores y modelos reutilizables, siguiendo los patrones de scikit-learn.

---
## 🗺️ Contenido
1. Clases y objetos: repaso con enfoque en DS
2. Herencia: construyendo jerarquías de transformadores
3. Métodos especiales (`__dunder__`): objetos que se comportan como arrays
4. Properties y descriptores: encapsulamiento inteligente
5. Patrón Transformer de scikit-learn: `fit`, `transform`, `fit_transform`
6. Dataclasses y NamedTuples: registros modernos de datos
7. Caso aplicado: Pipeline personalizado de preprocesamiento

---
## 1. Clases en DS: Encapsulando Estado y Comportamiento

### 📖 Concepto
En Machine Learning, los **objetos** encapsulan:
- **Estado aprendido** (`fit`): parámetros calculados sobre datos de entrenamiento (media, std, pesos...)
- **Transformaciones** (`transform`): aplicar el estado aprendido a nuevos datos
- **Validaciones**: evitar transformar antes de entrenar, verificar dimensiones

Este patrón es exactamente lo que usa `scikit-learn`.

In [ ]:
import numpy as np
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
import time

# ── Clase Base Abstracta: interfaz del transformador ─────────────────────
class BaseTransformer(ABC):
    """Clase base abstracta para transformadores de datos.
    
    Sigue el patrón Estimator de scikit-learn:
    - fit(X): aprende parámetros de X
    - transform(X): aplica transformación
    - fit_transform(X): fit + transform en un paso
    """

    def __init__(self):
        self._is_fitted = False
        self._fit_time  = None

    @abstractmethod
    def fit(self, X: np.ndarray, y=None) -> "BaseTransformer":
        """Aprende parámetros del dataset de entrenamiento."""
        ...

    @abstractmethod
    def transform(self, X: np.ndarray) -> np.ndarray:
        """Aplica la transformación aprendida."""
        ...

    def fit_transform(self, X: np.ndarray, y=None) -> np.ndarray:
        """Método de conveniencia: fit + transform."""
        return self.fit(X, y).transform(X)

    def _check_fitted(self):
        """Valida que el transformador ha sido entrenado."""
        if not self._is_fitted:
            raise RuntimeError(
                f"{self.__class__.__name__} debe ser entrenado antes de transformar. "
                "Llama a .fit() primero."
            )

    def __repr__(self) -> str:
        params = ", ".join(f"{k}={v}" for k, v in self.get_params().items())
        status = "[fitted]" if self._is_fitted else "[not fitted]"
        return f"{self.__class__.__name__}({params}) {status}"

    def get_params(self) -> Dict[str, Any]:
        """Retorna hiperparámetros — subclases deben sobreescribir si necesario."""
        return {}


print("Clase BaseTransformer definida correctamente")
print(f"Métodos abstractos: {BaseTransformer.__abstractmethods__}")

In [ ]:
# ── StandardScaler personalizado ──────────────────────────────────────────
class StandardScaler(BaseTransformer):
    """Estandarización Z-Score con soporte para features robustos.
    
    Attributes:
        with_mean (bool): centrar en media
        with_std  (bool): escalar por desviación estándar
        eps (float): estabilidad numérica
    """

    def __init__(self, with_mean: bool = True,
                 with_std: bool = True, eps: float = 1e-8):
        super().__init__()
        self.with_mean = with_mean
        self.with_std  = with_std
        self.eps       = eps
        # Parámetros aprendidos (privados hasta fit)
        self._mean_  = None
        self._scale_ = None
        self._n_features_ = None

    def fit(self, X: np.ndarray, y=None) -> "StandardScaler":
        X = np.asarray(X, dtype=np.float64)
        t0 = time.time()
        self._n_features_ = X.shape[1] if X.ndim > 1 else 1
        self._mean_  = X.mean(axis=0) if self.with_mean else np.zeros(self._n_features_)
        self._scale_ = X.std(axis=0)  if self.with_std  else np.ones(self._n_features_)
        self._is_fitted = True
        self._fit_time  = time.time() - t0
        return self  # permite encadenamiento: scaler.fit(X).transform(X_test)

    def transform(self, X: np.ndarray) -> np.ndarray:
        self._check_fitted()
        X = np.asarray(X, dtype=np.float64)
        if X.ndim > 1 and X.shape[1] != self._n_features_:
            raise ValueError(
                f"Esperaba {self._n_features_} features, recibió {X.shape[1]}"
            )
        return (X - self._mean_) / (self._scale_ + self.eps)

    def inverse_transform(self, X_scaled: np.ndarray) -> np.ndarray:
        """Revertir la transformación — útil para interpretar predicciones."""
        self._check_fitted()
        return X_scaled * (self._scale_ + self.eps) + self._mean_

    @property
    def mean_(self) -> np.ndarray:
        """Media aprendida durante fit (read-only)."""
        self._check_fitted()
        return self._mean_

    @property
    def scale_(self) -> np.ndarray:
        """Desviación estándar aprendida (read-only)."""
        self._check_fitted()
        return self._scale_

    def get_params(self) -> Dict[str, Any]:
        return {"with_mean": self.with_mean,
                "with_std": self.with_std, "eps": self.eps}


# ── Test del StandardScaler ───────────────────────────────────────────────
np.random.seed(42)
X_train = np.random.randn(1000, 5) * np.array([10, 0.1, 100, 1, 50])
X_test  = np.random.randn(200, 5)  * np.array([10, 0.1, 100, 1, 50])

scaler = StandardScaler()
print(scaler)  # __repr__

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # usa parámetros del train

print(scaler)  # ahora dice [fitted]
print(f"\nMedia train (debe ser ≈0): {X_train_scaled.mean(axis=0).round(4)}")
print(f"Std  train (debe ser ≈1):  {X_train_scaled.std(axis=0).round(4)}")
print(f"\n⚠️  Media test (no centrado en 0, correcto): {X_test_scaled.mean(axis=0).round(4)}")

# Test de error antes de fit
try:
    scaler2 = StandardScaler()
    scaler2.transform(X_test)
except RuntimeError as e:
    print(f"\n✅ Error correctamente capturado: {e}")

In [ ]:
# ── Herencia: QuantileTransformer (transformación robusta) ────────────────
class RobustScaler(BaseTransformer):
    """Escalado robusto usando mediana e IQR — resistente a outliers.
    
    Esencial en datos financieros, médicos o industriales con outliers extremos.
    """

    def __init__(self, quantile_range: tuple = (25.0, 75.0)):
        super().__init__()
        self.quantile_range = quantile_range
        self._center_ = None
        self._scale_  = None

    def fit(self, X: np.ndarray, y=None) -> "RobustScaler":
        X = np.asarray(X, dtype=np.float64)
        q_low, q_high = self.quantile_range
        self._center_ = np.median(X, axis=0)
        self._scale_  = (
            np.percentile(X, q_high, axis=0) -
            np.percentile(X, q_low, axis=0)
        )
        self._scale_[self._scale_ == 0] = 1.0  # evitar división por cero
        self._is_fitted = True
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        self._check_fitted()
        return (np.asarray(X, dtype=np.float64) - self._center_) / self._scale_

    def get_params(self):
        return {"quantile_range": self.quantile_range}


# Comparación StandardScaler vs RobustScaler con datos ruidosos
np.random.seed(42)
X_clean = np.random.randn(100, 3) * 5 + 10
# Agregar outliers extremos (simular errores de sensor)
X_outliers = X_clean.copy()
X_outliers[0] = [1000, -500, 2000]  # outliers masivos

std_scaler    = StandardScaler()
robust_scaler = RobustScaler()

X_std    = std_scaler.fit_transform(X_outliers)
X_robust = robust_scaler.fit_transform(X_outliers)

print("Impacto de outliers en escalado (feature 0):")
print(f"  StandardScaler — rango: [{X_std[:,0].min():.1f}, {X_std[:,0].max():.1f}]")
print(f"  RobustScaler   — rango: [{X_robust[:,0].min():.1f}, {X_robust[:,0].max():.1f}]")
print("\n→ RobustScaler contiene el outlier, StandardScaler lo amplifica")

In [ ]:
# ── Dataclasses: Configuración moderna de modelos ─────────────────────────
from dataclasses import dataclass, field
from typing import Optional, List

@dataclass
class ModelConfig:
    """Configuración de un modelo de ML — inmutable y serializable."""
    # Identificación
    nombre: str
    version: str = "1.0.0"

    # Arquitectura
    n_capas: int = 3
    n_neuronas: List[int] = field(default_factory=lambda: [128, 64, 32])
    activacion: str = "relu"
    dropout_rate: float = 0.3

    # Entrenamiento
    learning_rate: float = 1e-3
    batch_size: int = 32
    epochs: int = 100
    early_stopping_patience: int = 10

    # Opcional
    seed: Optional[int] = 42
    notas: str = ""

    def __post_init__(self):
        """Validaciones automáticas al crear el objeto."""
        if len(self.n_neuronas) != self.n_capas:
            raise ValueError(
                f"n_capas={self.n_capas} pero n_neuronas tiene {len(self.n_neuronas)} elementos"
            )
        if not 0 <= self.dropout_rate < 1:
            raise ValueError(f"dropout_rate debe estar en [0, 1), recibido: {self.dropout_rate}")
        if self.learning_rate <= 0:
            raise ValueError(f"learning_rate debe ser positivo")

    @property
    def total_parametros_estimados(self) -> int:
        """Estimación rápida del número de parámetros entrenables."""
        total = 0
        capas = self.n_neuronas
        for i in range(1, len(capas)):
            total += capas[i-1] * capas[i] + capas[i]  # pesos + biases
        return total


# Uso
config_churn = ModelConfig(
    nombre="ChurnPredictor_v2",
    n_capas=4,
    n_neuronas=[256, 128, 64, 1],
    dropout_rate=0.4,
    learning_rate=5e-4,
    notas="Modelo para predicción de churn en Telco. Dataset 2025Q1."
)

print("Configuración del modelo:")
print(config_churn)
print(f"\nParámetros estimados: {config_churn.total_parametros_estimados:,}")

# Test de validación automática
try:
    config_invalido = ModelConfig(
        nombre="Mal configurado",
        n_capas=3,
        n_neuronas=[128, 64]  # faltan neuronas
    )
except ValueError as e:
    print(f"\n✅ Validación correcta: {e}")

In [ ]:
# ── Pipeline personalizado: composición de transformadores ───────────────
class Pipeline:
    """Pipeline secuencial de transformadores — similar a sklearn.Pipeline.
    
    Permite encadenar transformaciones de forma legible y reproducible.
    
    Example:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('robust', RobustScaler()),
        ])
        X_out = pipe.fit_transform(X_train)
    """

    def __init__(self, steps: List[tuple]):
        self.steps = steps
        self._is_fitted = False
        self._validate_steps()

    def _validate_steps(self):
        names = [name for name, _ in self.steps]
        if len(names) != len(set(names)):
            raise ValueError("Los nombres de los pasos deben ser únicos")
        for name, transformer in self.steps:
            if not isinstance(transformer, BaseTransformer):
                raise TypeError(f"Paso '{name}' no es un BaseTransformer")

    def fit(self, X: np.ndarray, y=None) -> "Pipeline":
        X_current = X
        for name, transformer in self.steps:
            print(f"  Fitting '{name}'...")
            X_current = transformer.fit_transform(X_current, y)
        self._is_fitted = True
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        if not self._is_fitted:
            raise RuntimeError("Pipeline no entrenado")
        for _, transformer in self.steps:
            X = transformer.transform(X)
        return X

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

    def __len__(self) -> int:
        return len(self.steps)

    def __getitem__(self, name: str) -> BaseTransformer:
        """Acceso por nombre: pipeline['scaler']"""
        for step_name, transformer in self.steps:
            if step_name == name:
                return transformer
        raise KeyError(f"Paso '{name}' no encontrado en el pipeline")

    def __repr__(self):
        steps_str = "\n  ".join(f"{i+1}. {n}: {t}" for i, (n, t) in enumerate(self.steps))
        return f"Pipeline ({len(self)} pasos):\n  {steps_str}"


# Ejemplo: Pipeline de preprocesamiento para datos médicos
np.random.seed(42)
X_medico = np.random.randn(500, 8) * np.array([1, 100, 0.5, 50, 10, 1, 200, 5])
X_medico[np.random.rand(*X_medico.shape) < 0.02] *= 10  # outliers aleatorios

pipeline_medico = Pipeline([
    ("robust_scaler", RobustScaler(quantile_range=(10, 90))),
    ("standard_scaler", StandardScaler()),
])

print(pipeline_medico)
print("\nEntrenando pipeline:")
X_procesado = pipeline_medico.fit_transform(X_medico)

print(f"\nResultado:")
print(f"  Input  — std por feature: {X_medico.std(axis=0).round(1)}")
print(f"  Output — std por feature: {X_procesado.std(axis=0).round(3)}")

# Acceso a componentes del pipeline
print(f"\nMediana del RobustScaler: {pipeline_medico['robust_scaler']._center_.round(2)}")

---
## 🧠 Resumen de OOP para Data Science

### Patrones Esenciales

| Patrón | Cuándo Usarlo | Ejemplo en ML |
|--------|--------------|---------------|
| **Clase Abstracta (ABC)** | Definir interfaces | `BaseTransformer`, `BaseEstimator` |
| **Herencia** | Especializar comportamiento | `StandardScaler` → `RobustScaler` |
| **Properties** | Encapsular estado aprendido | `scaler.mean_`, `model.coef_` |
| **`__repr__`** | Debug en notebooks | Ver estado del objeto |
| **`__len__`, `__getitem__`** | Comportamiento de contenedor | `pipeline['step']`, `len(pipeline)` |
| **Dataclasses** | Configuración inmutable | `ModelConfig`, `TrainingArgs` |
| **`__post_init__`** | Validación automática | Verificar hiperparámetros |

### ✅ El patrón `fit / transform` es el estándar de la industria:
- `fit(X_train)` → aprende solo del train (evita **data leakage**)
- `transform(X_test)` → aplica sin re-aprender
- Nunca usar `fit_transform` en test — es el error más común